# CrowdSight: Face Recognition in Crowded Area Video (Google Colab)

**Vivek Shivhare (2025DSS1032) · Abhishek Dewangan (2025DSS1002)**
M.Sc. Data Science & Management · IIT Ropar

This notebook runs the full live dashboard inside Colab:

**SCRFD detection → 5-point alignment → ArcFace embedding → cosine matching → ByteTrack-style tracking with voting → FastAPI dashboard**

**How to use:** run the cells in order (Runtime → Run all works too).

1. **Install** the libraries (about 1 minute).
2. **Write the project code.** Every module is in this notebook, nothing to upload.
3. **Upload data:** a crowd video and a zip of gallery photos (one folder per person).
4. **Enroll** the gallery.
5. **Start** the dashboard and open it in a new tab or right here.

> **Colab note:** Colab runs on a Google server, so it can't use your laptop's webcam. Use an uploaded **video** as the source, and enroll people with **＋ Enroll → Upload photos**. Everything else works: tracking, recognition, watchlist alerts, Find person, privacy blur, heatmap and CSV export.


## 1 · Install libraries

In [ ]:
%%capture
!pip install -q insightface onnxruntime fastapi uvicorn python-multipart opencv-python-headless
# Optional, for a GPU runtime: replace onnxruntime with onnxruntime-gpu. The code uses CUDA automatically if it is available.

## 2 · Write the project code
Each cell below writes one file of the project, unchanged from the version that runs on the laptop.
Key files: `detector.py` (SCRFD + tiling), `recognizer.py` (ArcFace + gallery), `tracker.py` (tracks + voting), `pipeline.py` (detect → track → recognise), `server.py` (dashboard backend), `index.html` (dashboard UI).

In [ ]:
import os
PROJECT = "/content/crowd-face-recognition"
for d in ["src", "app/static", "scripts", "data/gallery", "data/videos", "data/images", "outputs"]:
    os.makedirs(f"{PROJECT}/{d}", exist_ok=True)
%cd {PROJECT}
print("Project folder ready:", PROJECT)

In [ ]:
%%writefile src/__init__.py
# package marker


In [ ]:
%%writefile src/config.py
"""Central settings for the project. Change values here, not inside scripts."""
import os
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"
GALLERY_DIR = DATA_DIR / "gallery"      # one sub-folder per enrolled person
VIDEO_DIR = DATA_DIR / "videos"
IMAGE_DIR = DATA_DIR / "images"
OUTPUT_DIR = ROOT / "outputs"

# InsightFace model pack: "buffalo_l" (accurate) or "buffalo_s" (faster, for weak laptops)
MODEL_PACK = os.environ.get("FACE_MODEL", "buffalo_l")
# NOTE: after changing MODEL_PACK, re-run scripts/enroll.py (embeddings differ between packs)

# Detection
DET_SIZE = 640            # detector input size (pixels). Higher = finds smaller faces, slower
DET_THRESHOLD = 0.5       # minimum detection confidence
MIN_FACE_SIZE = 12        # ignore faces smaller than this (pixels, shorter side)

# Tiling (helps with tiny faces in crowds)
TILE_SIZE = 640
TILE_OVERLAP = 0.25
NMS_IOU = 0.4

# Recognition
MATCH_THRESHOLD = 0.45    # cosine similarity for a known person. Canteen video: own-face median 0.59, wrong matches <= 0.20

# Video
PROCESS_EVERY = 1         # run the model on every Nth frame (2-3 = faster on CPU)


In [ ]:
%%writefile src/detector.py
"""Face detector for crowded scenes (SCRFD via InsightFace), with optional tiling.

Tiling: a crowd frame is split into overlapping tiles, each tile is run through the
detector at full resolution, and boxes are merged with non-max suppression. This finds
small, distant faces that are lost when the whole frame is shrunk to 640x640.
"""
from __future__ import annotations

import numpy as np
import onnxruntime as ort
from insightface.app import FaceAnalysis

from src import config


def pick_providers() -> list[str]:
    """Use the GPU automatically if onnxruntime-gpu and CUDA are available."""
    available = ort.get_available_providers()
    if "CUDAExecutionProvider" in available:
        return ["CUDAExecutionProvider", "CPUExecutionProvider"]
    return ["CPUExecutionProvider"]


def nms(boxes: np.ndarray, scores: np.ndarray, iou_thr: float) -> list[int]:
    """Standard non-max suppression. boxes: (N,4) x1,y1,x2,y2."""
    if len(boxes) == 0:
        return []
    x1, y1, x2, y2 = boxes.T
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    keep = []
    while order.size:
        i = order[0]
        keep.append(int(i))
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
        iou = inter / (areas[i] + areas[order[1:]] - inter + 1e-9)
        order = order[1:][iou <= iou_thr]
    return keep


class FaceDetector:
    def __init__(self, det_size: int = config.DET_SIZE, det_thresh: float = config.DET_THRESHOLD):
        self.providers = pick_providers()
        # Detection-only: skips recognition/landmark models, so it is fast.
        self.app = FaceAnalysis(name=config.MODEL_PACK, allowed_modules=["detection"],
                                providers=self.providers)
        ctx = 0 if self.providers[0].startswith("CUDA") else -1
        self.app.prepare(ctx_id=ctx, det_size=(det_size, det_size), det_thresh=det_thresh)
        self.det = self.app.det_model
        self.det_size = det_size

    @property
    def device(self) -> str:
        return "GPU (CUDA)" if self.providers[0].startswith("CUDA") else "CPU"

    def _detect(self, img: np.ndarray):
        bboxes, kpss = self.det.detect(img, input_size=(self.det_size, self.det_size))
        if kpss is None:
            kpss = np.zeros((len(bboxes), 5, 2), dtype=np.float32)
        return bboxes, kpss

    def detect(self, img: np.ndarray, tile: bool = False):
        """Returns (boxes (N,4), scores (N,), landmarks (N,5,2))."""
        if not tile:
            bboxes, kpss = self._detect(img)
        else:
            bboxes, kpss = self._detect_tiled(img)
        if len(bboxes) == 0:
            return np.zeros((0, 4)), np.zeros(0), np.zeros((0, 5, 2))
        boxes, scores = bboxes[:, :4], bboxes[:, 4]
        w = boxes[:, 2] - boxes[:, 0]
        h = boxes[:, 3] - boxes[:, 1]
        ok = np.minimum(w, h) >= config.MIN_FACE_SIZE
        return boxes[ok], scores[ok], kpss[ok]

    def _detect_tiled(self, img: np.ndarray):
        H, W = img.shape[:2]
        ts = config.TILE_SIZE
        step = int(ts * (1 - config.TILE_OVERLAP))
        all_b, all_k = [], []

        # 1) Whole frame once, for large/near faces that tiles might cut in half.
        b, k = self._detect(img)
        all_b.append(b); all_k.append(k)

        # 2) Overlapping tiles at native resolution, for small/far faces.
        if max(H, W) > ts:
            ys = list(range(0, max(H - ts, 0) + 1, step)) or [0]
            xs = list(range(0, max(W - ts, 0) + 1, step)) or [0]
            if ys[-1] + ts < H: ys.append(H - ts)
            if xs[-1] + ts < W: xs.append(W - ts)
            for y in ys:
                for x in xs:
                    crop = img[max(y, 0):y + ts, max(x, 0):x + ts]
                    b, k = self._detect(crop)
                    if len(b):
                        b = b.copy(); k = k.copy()
                        b[:, [0, 2]] += max(x, 0); b[:, [1, 3]] += max(y, 0)
                        k[:, :, 0] += max(x, 0); k[:, :, 1] += max(y, 0)
                        all_b.append(b); all_k.append(k)

        bboxes = np.concatenate(all_b) if all_b else np.zeros((0, 5))
        kpss = np.concatenate(all_k) if all_k else np.zeros((0, 5, 2))
        keep = nms(bboxes[:, :4], bboxes[:, 4], config.NMS_IOU)
        return bboxes[keep], kpss[keep]


def draw_detections(img: np.ndarray, boxes, scores, kpss=None) -> np.ndarray:
    import cv2
    out = img.copy()
    for i, (b, s) in enumerate(zip(boxes, scores)):
        x1, y1, x2, y2 = b.astype(int)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 220, 0), 2)
        cv2.putText(out, f"{s:.2f}", (x1, max(y1 - 4, 10)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, (0, 220, 0), 1, cv2.LINE_AA)
        if kpss is not None:
            for (px, py) in kpss[i].astype(int):
                cv2.circle(out, (px, py), 1, (0, 0, 255), -1)
    label = f"Faces: {len(boxes)}"
    cv2.rectangle(out, (0, 0), (170, 34), (0, 0, 0), -1)
    cv2.putText(out, label, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    return out


In [ ]:
%%writefile src/recognizer.py
"""Face recognition: ArcFace embeddings + gallery matching.

Gallery layout:  data/gallery/<PersonName>/*.jpg   (5–10 photos per person)
Enrollment builds data/gallery_embeddings.npz with one averaged, L2-normalised
512-d embedding per person. Matching uses cosine similarity.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import cv2
import numpy as np
from insightface.app.common import Face
from insightface.model_zoo import get_model
from insightface.utils import face_align

from src import config
from src.detector import FaceDetector

GALLERY_FILE = config.DATA_DIR / "gallery_embeddings.npz"
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def l2norm(x: np.ndarray) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-9)


@dataclass
class FaceResult:
    bbox: np.ndarray
    score: float
    kps: np.ndarray
    embedding: np.ndarray | None = None
    name: str = "Unknown"
    similarity: float = 0.0
    track_id: int | None = None
    extra: dict = field(default_factory=dict)


class Gallery:
    def __init__(self):
        self.names: list[str] = []
        self.embs = np.zeros((0, 512), dtype=np.float32)

    def __len__(self):
        return len(self.names)

    def add(self, name: str, emb: np.ndarray):
        emb = l2norm(emb.astype(np.float32))
        if name in self.names:  # update: average with existing
            i = self.names.index(name)
            self.embs[i] = l2norm(self.embs[i] + emb)
        else:
            self.names.append(name)
            self.embs = np.vstack([self.embs, emb[None]])

    def remove(self, name: str):
        if name in self.names:
            i = self.names.index(name)
            self.names.pop(i)
            self.embs = np.delete(self.embs, i, axis=0)

    def match(self, emb: np.ndarray, threshold: float):
        if len(self) == 0:
            return "Unknown", 0.0
        sims = self.embs @ l2norm(emb)
        i = int(sims.argmax())
        s = float(sims[i])
        return (self.names[i] if s >= threshold else "Unknown"), s

    def save(self, path: Path = GALLERY_FILE):
        np.savez(path, names=np.array(self.names), embs=self.embs, model=np.array(config.MODEL_PACK))

    @classmethod
    def load(cls, path: Path = GALLERY_FILE) -> "Gallery":
        g = cls()
        if Path(path).exists():
            d = np.load(path)
            g.names = [str(n) for n in d["names"]]
            g.embs = d["embs"].astype(np.float32)
            built = str(d["model"]) if "model" in d else config.MODEL_PACK
            if built != config.MODEL_PACK:
                print(f"WARNING: gallery was built with {built} but current model is "
                      f"{config.MODEL_PACK}. Re-run scripts/enroll.py.")
                return cls()
        return g


class FaceRecognizer:
    """Detector + ArcFace in one object. Works with tiled detection too."""

    def __init__(self, det_size: int = config.DET_SIZE, threshold: float = config.MATCH_THRESHOLD):
        self.detector = FaceDetector(det_size=det_size)
        model_dir = Path.home() / ".insightface" / "models" / config.MODEL_PACK
        rec_path = next(p for p in model_dir.glob("*.onnx") if p.name.startswith("w600k"))
        self.rec = get_model(str(rec_path), providers=self.detector.providers)
        self.rec.prepare(ctx_id=0 if self.detector.device.startswith("GPU") else -1)
        self.threshold = threshold
        self.gallery = Gallery.load()

    @property
    def device(self):
        return self.detector.device

    def embed(self, img: np.ndarray, kps: np.ndarray) -> np.ndarray:
        """Align with 5 landmarks and return a normalised 512-d embedding."""
        f = Face(kps=kps)
        return l2norm(self.rec.get(img, f))

    def process(self, img: np.ndarray, tile: bool = False, recognize: bool = True) -> list[FaceResult]:
        boxes, scores, kpss = self.detector.detect(img, tile=tile)
        out = [FaceResult(bbox=b, score=float(s), kps=k) for b, s, k in zip(boxes, scores, kpss)]
        if recognize and out:
            embs = self.embed_batch(img, [r.kps for r in out])
            for r, e in zip(out, embs):
                r.embedding = e
                r.name, r.similarity = self.gallery.match(e, self.threshold)
        return out

    def embed_batch(self, img: np.ndarray, kps_list) -> np.ndarray:
        """Align all faces and embed them in one batched model call."""
        crops = [face_align.norm_crop(img, k, 112) for k in kps_list]
        return l2norm(self.rec.get_feat(crops))

    # ---------- enrollment ----------
    def embed_largest_face(self, img: np.ndarray) -> np.ndarray | None:
        boxes, scores, kpss = self.detector.detect(img)
        if len(boxes) == 0:
            return None
        areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        return self.embed(img, kpss[int(areas.argmax())])

    def enroll_folder(self, gallery_dir: Path = config.GALLERY_DIR, verbose=True) -> Gallery:
        g = Gallery()
        for person in sorted(p for p in Path(gallery_dir).iterdir() if p.is_dir()):
            embs = []
            for f in sorted(person.iterdir()):
                if f.suffix.lower() not in IMG_EXT:
                    continue
                img = cv2.imread(str(f))
                e = None if img is None else self.embed_largest_face(img)
                if e is None:
                    if verbose:
                        print(f"   ! no face found in {f.name}, skipped")
                    continue
                embs.append(e)
            if embs:
                g.add(person.name, np.mean(embs, axis=0))
                if verbose:
                    print(f"  + {person.name:<20} {len(embs)} photo(s)")
        g.save()
        self.gallery = g
        return g


# ---------- drawing ----------
def color_for(name: str):
    if name == "Unknown":
        return (60, 60, 230)  # red (BGR)
    h = abs(hash(name)) % 180
    c = cv2.cvtColor(np.uint8([[[h, 200, 230]]]), cv2.COLOR_HSV2BGR)[0, 0]
    return tuple(int(v) for v in c)


def draw_results(img: np.ndarray, results: list[FaceResult], show_unknown_label=False,
                 blur_unknown=False) -> np.ndarray:
    out = img.copy()
    for r in results:
        x1, y1, x2, y2 = r.bbox.astype(int)
        x1, y1 = max(x1, 0), max(y1, 0)
        if blur_unknown and r.name == "Unknown" and x2 > x1 and y2 > y1:
            roi = out[y1:y2, x1:x2]
            k = max(3, ((x2 - x1) // 3) | 1)
            out[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (k, k), 0)
        col = color_for(r.name)
        cv2.rectangle(out, (x1, y1), (x2, y2), col, 2)
        if r.name != "Unknown" or show_unknown_label:
            label = r.name if r.name == "Unknown" else f"{r.name} {r.similarity:.2f}"
            if r.track_id is not None:
                label = f"#{r.track_id} {label}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(out, (x1, y1 - th - 6), (x1 + tw + 4, y1), col, -1)
            cv2.putText(out, label, (x1 + 2, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                        (255, 255, 255), 1, cv2.LINE_AA)
    return out


In [ ]:
%%writefile src/tracker.py
"""Lightweight multi-face tracker (ByteTrack-style, no extra dependencies).

Why tracking matters in crowds
  * Each face gets a stable ID (#12) across frames.
  * Recognition runs once per track every few frames instead of on every face every
    frame -> big speed-up.
  * The identity of a track is a vote over many frames -> names stop flickering,
    and one bad frame (blur, side view) can't cause a wrong name.

Matching: high-confidence detections are matched to existing tracks first by IoU,
then low-confidence detections get a second chance (the ByteTrack idea), which keeps
tracks alive through partial occlusion. A constant-velocity motion model predicts
where each box moves between frames.
"""
from __future__ import annotations

from collections import Counter

import numpy as np


def iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def greedy_match(iou: np.ndarray, thr: float):
    """Return (pairs, unmatched_rows, unmatched_cols) using greedy highest-IoU-first."""
    pairs, used_r, used_c = [], set(), set()
    if iou.size:
        order = np.dstack(np.unravel_index(np.argsort(-iou, axis=None), iou.shape))[0]
        for r, c in order:
            if iou[r, c] < thr:
                break
            if r in used_r or c in used_c:
                continue
            pairs.append((int(r), int(c)))
            used_r.add(int(r)); used_c.add(int(c))
    un_r = [i for i in range(iou.shape[0]) if i not in used_r]
    un_c = [j for j in range(iou.shape[1]) if j not in used_c]
    return pairs, un_r, un_c


class Track:
    _next_id = 1

    def __init__(self, bbox, score, kps, frame_idx):
        self.id = Track._next_id
        Track._next_id += 1
        self.bbox = np.asarray(bbox, dtype=np.float32)
        self.velocity = np.zeros(4, dtype=np.float32)
        self.score = score
        self.kps = kps
        self.hits = 1
        self.age = 0
        self.misses = 0
        self.first_frame = frame_idx
        self.last_frame = frame_idx
        self.votes: Counter = Counter()      # name -> summed similarity
        self.best_sim: dict = {}             # name -> best similarity seen
        self.last_rec_frame = -10**9
        self.n_rec = 0
        self.matched_this_frame = True

    # ----- motion -----
    def predict(self):
        self.bbox = self.bbox + self.velocity
        self.age += 1

    def update(self, bbox, score, kps, frame_idx):
        bbox = np.asarray(bbox, dtype=np.float32)
        self.velocity = 0.6 * self.velocity + 0.4 * (bbox - self.bbox)
        self.bbox = bbox
        self.score, self.kps = score, kps
        self.hits += 1
        self.misses = 0
        self.last_frame = frame_idx
        self.matched_this_frame = True

    # ----- identity voting -----
    def add_vote(self, name: str, sim: float):
        self.n_rec += 1
        self.votes[name] += max(sim, 0.01)
        self.best_sim[name] = max(self.best_sim.get(name, 0.0), sim)

    @property
    def name(self) -> str:
        if not self.votes:
            return "Unknown"
        total = sum(self.votes.values())
        name, w = self.votes.most_common(1)[0]
        # a name must hold the majority of the vote weight
        return name if w / total >= 0.5 else "Unknown"

    @property
    def similarity(self) -> float:
        return self.best_sim.get(self.name, 0.0)

    @property
    def confirmed(self) -> bool:
        return self.hits >= 2


class FaceTracker:
    def __init__(self, high_thr=0.6, iou_thr=0.3, low_iou_thr=0.2, max_misses=15):
        self.tracks: list[Track] = []
        self.high_thr, self.iou_thr, self.low_iou_thr = high_thr, iou_thr, low_iou_thr
        self.max_misses = max_misses
        self.frame_idx = 0

    def reset(self):
        self.tracks = []
        self.frame_idx = 0

    def update(self, boxes: np.ndarray, scores: np.ndarray, kpss: np.ndarray) -> list[Track]:
        """Feed one frame of detections. Returns tracks matched in this frame."""
        self.frame_idx += 1
        for t in self.tracks:
            t.predict()
            t.matched_this_frame = False

        high = np.where(scores >= self.high_thr)[0]
        low = np.where(scores < self.high_thr)[0]
        tb = np.array([t.bbox for t in self.tracks]) if self.tracks else np.zeros((0, 4))

        # 1st association: high-confidence detections
        pairs, un_t, un_d = greedy_match(iou_matrix(tb, boxes[high]), self.iou_thr)
        for ti, di in pairs:
            d = high[di]
            self.tracks[ti].update(boxes[d], scores[d], kpss[d], self.frame_idx)
        new_dets = [high[i] for i in un_d]

        # 2nd association: low-confidence detections with leftover tracks
        rest = [self.tracks[i] for i in un_t]
        if rest and len(low):
            rb = np.array([t.bbox for t in rest])
            pairs2, _, _ = greedy_match(iou_matrix(rb, boxes[low]), self.low_iou_thr)
            for ti, di in pairs2:
                d = low[di]
                rest[ti].update(boxes[d], scores[d], kpss[d], self.frame_idx)

        for t in self.tracks:
            if not t.matched_this_frame:
                t.misses += 1
                t.velocity *= 0.5
        self.tracks = [t for t in self.tracks if t.misses <= self.max_misses]

        for d in new_dets:
            self.tracks.append(Track(boxes[d], scores[d], kpss[d], self.frame_idx))

        return [t for t in self.tracks if t.matched_this_frame]


In [ ]:
%%writefile src/pipeline.py
"""Full crowd pipeline: detect -> track -> recognize (per track, with voting) -> events.

Used by the live dashboard (app/server.py) and scripts/track_video.py.
"""
from __future__ import annotations

import csv
import io
import time
from collections import deque
from dataclasses import dataclass, field

import cv2
import numpy as np

from src import config
from src.recognizer import FaceRecognizer, color_for
from src.tracker import FaceTracker, Track


@dataclass
class Sighting:
    name: str
    first_seen: float
    last_seen: float
    frames: int = 0
    best_sim: float = 0.0
    snapshot: bytes | None = None      # JPEG of the best face crop
    tracks: set = field(default_factory=set)


@dataclass
class Settings:
    threshold: float = config.MATCH_THRESHOLD
    privacy: bool = False             # blur unknown faces
    tile: bool = False                # tiled detection for tiny faces
    detect_every: int = 1             # run the detector every N frames (tracker fills the gaps)
    recheck_every: int = 15           # re-recognize a known track every N frames
    max_rec_per_frame: int = 12       # cap recognitions per frame (keeps FPS steady in big crowds)
    show_ids: bool = True
    show_unknown_label: bool = False


class CrowdPipeline:
    def __init__(self, recognizer: FaceRecognizer | None = None, settings: Settings | None = None):
        self.rec = recognizer or FaceRecognizer()
        self.s = settings or Settings()
        self.tracker = FaceTracker()
        self.watchlist: set[str] = set()
        self.reset_session()

    # ------------------------------------------------------------------ session
    def reset_session(self):
        self.tracker.reset()
        self.sightings: dict[str, Sighting] = {}
        self.alerts: deque = deque(maxlen=50)
        self.events: deque = deque(maxlen=200)
        self.history: deque = deque(maxlen=300)   # (t, faces, known)
        self.alerted_tracks: set = set()
        self.last_alert: dict = {}
        self.alert_cooldown = 20.0      # seconds between alerts for the same person
        self.heat = np.zeros((36, 64), dtype=np.float32)
        self.frame_no = 0
        self.t_start = time.time()
        self.last_tracks: list[Track] = []
        self.fps = 0.0
        self.unique_tracks: set = set()      # unknown tracks seen for >= MIN_HITS frames
        self.named_tracks: set = set()       # tracks that were ever given a name
        self.last_ms = {"detect": 0.0, "recognize": 0.0}

    # ------------------------------------------------------------------ core
    def step(self, frame: np.ndarray) -> list[Track]:
        """Process one frame. Returns the visible tracks."""
        t0 = time.perf_counter()
        self.frame_no += 1
        run_det = (self.frame_no - 1) % max(1, self.s.detect_every) == 0

        if run_det:
            td = time.perf_counter()
            boxes, scores, kpss = self.rec.detector.detect(frame, tile=self.s.tile)
            self.last_ms["detect"] = (time.perf_counter() - td) * 1000
            tracks = self.tracker.update(boxes, scores, kpss)
            tr = time.perf_counter()
            self._recognize(frame, tracks)
            self.last_ms["recognize"] = (time.perf_counter() - tr) * 1000
        else:
            for t in self.tracker.tracks:
                t.predict()
            tracks = [t for t in self.tracker.tracks if t.misses == 0]

        visible = [t for t in tracks if t.confirmed]
        self._update_events(frame, visible)
        self.last_tracks = visible

        # FPS = frames actually shown per second of wall-clock time
        now_t = time.perf_counter()
        last = getattr(self, "_last_step_t", None)
        self._last_step_t = now_t
        if last is not None:
            inst = 1.0 / max(now_t - last, 1e-3)
            self.fps = 0.9 * self.fps + 0.1 * inst if self.fps else inst
        return visible

    def _recognize(self, frame: np.ndarray, tracks: list[Track]):
        if len(self.rec.gallery) == 0 or not tracks:
            return
        f = self.tracker.frame_idx
        due = []
        for t in tracks:
            since = f - t.last_rec_frame
            if t.n_rec < 3 and since >= 2:
                due.append(t)                      # new track: gather a few votes quickly
            elif t.name == "Unknown" and since >= 6:
                due.append(t)                      # unknown: retry (face may turn toward camera)
            elif since >= self.s.recheck_every:
                due.append(t)                      # known: periodic re-check
        if not due:
            return
        # prioritise tracks with fewest votes, then bigger faces (better quality)
        due.sort(key=lambda t: (t.n_rec, -(t.bbox[2] - t.bbox[0])))
        due = due[: self.s.max_rec_per_frame]
        embs = self.rec.embed_batch(frame, [t.kps for t in due])
        for t, e in zip(due, embs):
            name, sim = self.rec.gallery.match(e, self.s.threshold)
            t.add_vote(name, sim)
            t.last_rec_frame = f

    def _update_events(self, frame: np.ndarray, tracks: list[Track]):
        now = time.time()
        H, W = frame.shape[:2]
        known = 0
        for t in tracks:
            if t.name != "Unknown":
                self.named_tracks.add(t.id)
            elif t.hits >= 8:                # ~1 s of tracking: a real face, not a flicker
                self.unique_tracks.add(t.id)
            x1, y1, x2, y2 = t.bbox
            cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
            if 0 <= cx < 1 and 0 <= cy < 1:
                self.heat[int(cy * self.heat.shape[0]), int(cx * self.heat.shape[1])] += 1
            name = t.name
            if name == "Unknown":
                continue
            known += 1
            s = self.sightings.get(name)
            if s is None:
                s = self.sightings[name] = Sighting(name, now, now)
                self.events.appendleft({"time": now, "type": "seen", "name": name, "track": t.id})
            s.last_seen = now
            s.frames += 1
            s.tracks.add(t.id)
            if t.similarity > s.best_sim or s.snapshot is None:
                s.best_sim = t.similarity
                s.snapshot = self._crop_jpeg(frame, t.bbox)
            if (name in self.watchlist and t.id not in self.alerted_tracks
                    and now - self.last_alert.get(name, 0) > self.alert_cooldown):
                self.alerted_tracks.add(t.id)
                self.last_alert[name] = now
                self.alerts.appendleft({"id": len(self.alerted_tracks), "time": now, "name": name,
                                        "track": t.id, "similarity": round(t.similarity, 3)})
                self.events.appendleft({"time": now, "type": "alert", "name": name, "track": t.id})
        self.history.append((round(now - self.t_start, 1), len(tracks), known))

    @staticmethod
    def _crop_jpeg(frame, bbox, pad=0.35):
        x1, y1, x2, y2 = bbox
        w, h = x2 - x1, y2 - y1
        H, W = frame.shape[:2]
        a = [int(max(0, x1 - pad * w)), int(max(0, y1 - pad * h)),
             int(min(W, x2 + pad * w)), int(min(H, y2 + pad * h))]
        crop = frame[a[1]:a[3], a[0]:a[2]]
        if crop.size == 0:
            return None
        crop = cv2.resize(crop, (128, int(128 * crop.shape[0] / max(crop.shape[1], 1))))
        return cv2.imencode(".jpg", crop, [cv2.IMWRITE_JPEG_QUALITY, 85])[1].tobytes()

    # ------------------------------------------------------------------ drawing
    def draw(self, frame: np.ndarray, tracks: list[Track] | None = None) -> np.ndarray:
        tracks = self.last_tracks if tracks is None else tracks
        out = frame.copy()
        H, W = out.shape[:2]
        L = max(W, H)
        thick = max(2, int(round(L / 640)))
        fs = 0.5 * max(1.0, L / 960)
        for t in tracks:
            x1, y1, x2, y2 = [int(v) for v in t.bbox]
            x1, y1, x2, y2 = max(x1, 0), max(y1, 0), min(x2, W), min(y2, H)
            if x2 <= x1 or y2 <= y1:
                continue
            name = t.name
            if self.s.privacy and name == "Unknown":
                roi = out[y1:y2, x1:x2]
                k = max(3, ((x2 - x1) // 2) | 1)
                out[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (k, k), 0)
            col = (0, 0, 255) if name in self.watchlist else color_for(name)
            cv2.rectangle(out, (x1, y1), (x2, y2), col, thick + (1 if name in self.watchlist else 0))
            if name != "Unknown" or self.s.show_unknown_label or self.s.show_ids:
                parts = []
                if self.s.show_ids:
                    parts.append(f"#{t.id}")
                if name != "Unknown":
                    parts.append(f"{name} {t.similarity:.2f}")
                elif self.s.show_unknown_label:
                    parts.append("Unknown")
                label = " ".join(parts)
                if name == "Unknown" and not self.s.show_unknown_label and (x2 - x1) < 28:
                    continue  # tiny unknown faces: box only, keep the crowd view readable
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, fs, 2)
                cv2.rectangle(out, (x1, y1 - th - 6), (x1 + tw + 4, y1), col, -1)
                cv2.putText(out, label, (x1 + 2, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, fs,
                            (255, 255, 255), 2, cv2.LINE_AA)
        return out

    # ------------------------------------------------------------------ reporting
    def state(self) -> dict:
        tracks = self.last_tracks
        known = [t for t in tracks if t.name != "Unknown"]
        now = time.time()
        present = sorted({t.name for t in known})
        return {
            "faces": len(tracks),
            "known": len(known),
            "unknown": len(tracks) - len(known),
            # people seen = each recognised person once + unknown faces tracked for ~1 s or more
            "unique_faces": len(self.sightings) + len(self.unique_tracks - self.named_tracks),
            "fps": round(self.fps, 1),
            "ms": {k: round(v) for k, v in self.last_ms.items()},
            "present": present,
            "people": [
                {"name": s.name, "first_seen": s.first_seen, "last_seen": s.last_seen,
                 "seconds_ago": round(now - s.last_seen, 1), "frames": s.frames,
                 "best_sim": round(s.best_sim, 3), "watch": s.name in self.watchlist,
                 "present": s.name in present}
                for s in sorted(self.sightings.values(), key=lambda s: -s.last_seen)
            ],
            "events": list(self.events)[:40],
            "alerts": list(self.alerts)[:10],
            "history": list(self.history)[-120:],
            "gallery": list(self.rec.gallery.names),
            "watchlist": sorted(self.watchlist),
            "settings": self.s.__dict__,
            "uptime": round(now - self.t_start, 1),
        }

    def heatmap(self) -> list:
        h = self.heat
        if h.max() > 0:
            h = cv2.GaussianBlur(h, (0, 0), 1.2)
            h = h / h.max()
        return np.round(h, 2).tolist()

    def export_csv(self) -> str:
        buf = io.StringIO()
        w = csv.writer(buf)
        w.writerow(["name", "first_seen", "last_seen", "duration_s", "frames_seen",
                    "best_similarity", "on_watchlist"])
        for s in sorted(self.sightings.values(), key=lambda s: s.first_seen):
            w.writerow([s.name, time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(s.first_seen)),
                        time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(s.last_seen)),
                        round(s.last_seen - s.first_seen, 1), s.frames, round(s.best_sim, 3),
                        s.name in self.watchlist])
        return buf.getvalue()


In [ ]:
%%writefile src/search.py
"""'Find this person': scan a video for one face, given a single photo.

Returns every moment the person appears, grouped into time segments, with a thumbnail
for each segment. Used by the dashboard and scripts/find_person.py.
"""
from __future__ import annotations

import base64
from typing import Callable

import cv2
import numpy as np

from src.recognizer import FaceRecognizer


def scan_video(rec: FaceRecognizer, query_emb: np.ndarray, video_path: str, threshold: float,
               sample_fps: float = 4.0, tile: bool = False,
               progress: Callable[[float], None] | None = None,
               should_stop: Callable[[], bool] | None = None) -> dict:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    step = max(1, int(round(fps / sample_fps)))
    hits, best_overall = [], 0.0
    idx = 0
    while True:
        if should_stop and should_stop():
            break
        ok = cap.grab()
        if not ok:
            break
        if idx % step == 0:
            ok, frame = cap.retrieve()
            if not ok:
                break
            boxes, scores, kpss = rec.detector.detect(frame, tile=tile)
            if len(boxes):
                embs = rec.embed_batch(frame, list(kpss))
                sims = embs @ query_emb
                i = int(sims.argmax())
                best_overall = max(best_overall, float(sims[i]))
                if sims[i] >= threshold:
                    x1, y1, x2, y2 = boxes[i].astype(int)
                    pad = int(0.4 * (x2 - x1))
                    crop = frame[max(0, y1 - pad):y2 + pad, max(0, x1 - pad):x2 + pad]
                    thumb = None
                    if crop.size:
                        crop = cv2.resize(crop, (96, int(96 * crop.shape[0] / max(1, crop.shape[1]))))
                        thumb = base64.b64encode(cv2.imencode(".jpg", crop)[1]).decode()
                    hits.append({"t": round(idx / fps, 2), "sim": round(float(sims[i]), 3),
                                 "box": [int(v) for v in boxes[i]], "thumb": thumb})
            if progress:
                progress(min(1.0, idx / total))
        idx += 1
    cap.release()
    duration = idx / fps

    # group hits into segments (gap > 1.5 s starts a new segment)
    segments = []
    for h in hits:
        if segments and h["t"] - segments[-1]["end"] <= 1.5:
            seg = segments[-1]
            seg["end"] = h["t"]
            seg["n"] += 1
            if h["sim"] > seg["best_sim"]:
                seg["best_sim"], seg["thumb"] = h["sim"], h["thumb"]
        else:
            segments.append({"start": h["t"], "end": h["t"], "n": 1, "best_sim": h["sim"],
                             "thumb": h["thumb"]})
    for h in hits:
        h.pop("thumb", None)
    return {"duration": round(duration, 2), "hits": hits, "segments": segments,
            "best_similarity": round(best_overall, 3), "threshold": threshold,
            "found": bool(segments)}


In [ ]:
%%writefile app/server.py
"""Live interactive dashboard for crowd face recognition.

Run:   python app/server.py            then open http://localhost:8000
       python app/server.py --source data/videos/crowd.mp4
       python app/server.py --port 8080 --model buffalo_s

Architecture
  capture thread  -> always holds the newest frame (webcam or looping video file)
  worker thread   -> detect + track + recognize the newest frame, draw, encode JPEG
  FastAPI         -> MJPEG video stream, JSON state polling, REST actions
"""
from __future__ import annotations

import argparse
import os
import shutil
import sys
import threading
import time
import uuid
from pathlib import Path

import cv2
import numpy as np

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

_args_pre = argparse.ArgumentParser(add_help=False)
_args_pre.add_argument("--model")
_known, _ = _args_pre.parse_known_args()
if _known.model:
    os.environ["FACE_MODEL"] = _known.model

from fastapi import FastAPI, File, Form, HTTPException, UploadFile  # noqa: E402
from fastapi.responses import (FileResponse, JSONResponse, Response,  # noqa: E402
                               StreamingResponse)

from src import config  # noqa: E402
from src.pipeline import CrowdPipeline, Settings  # noqa: E402
from src.recognizer import IMG_EXT, FaceRecognizer, l2norm  # noqa: E402
from src.search import scan_video  # noqa: E402

VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}
STATIC = Path(__file__).resolve().parent / "static"


# =============================================================================== video
class VideoSource:
    """Background reader that always exposes the most recent frame."""

    def __init__(self):
        self.lock = threading.Lock()
        self.cap = None
        self.kind = "none"
        self.label = "No source"
        self.frame = None
        self.frame_id = 0
        self.fps = 25.0
        self.paused = False
        self.error = None
        self.width = self.height = 0
        self._stop = threading.Event()
        threading.Thread(target=self._loop, daemon=True).start()

    def open(self, src: str):
        cap = None
        if src.isdigit():
            idx = int(src)
            if sys.platform == "win32":
                cap = cv2.VideoCapture(idx, cv2.CAP_DSHOW)
                if not cap.isOpened():
                    cap = cv2.VideoCapture(idx)
            else:
                cap = cv2.VideoCapture(idx)
            if cap.isOpened():
                cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
                cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
            kind, label = "webcam", f"Webcam {idx}"
        else:
            p = Path(src)
            if not p.is_absolute():
                p = ROOT / p
            cap = cv2.VideoCapture(str(p))
            kind, label = "file", p.name
        if not cap or not cap.isOpened():
            raise RuntimeError(f"Could not open {src}")
        if kind == "webcam":
            ok = False
            for _ in range(15):              # cameras often need a few reads to warm up
                ok, _f = cap.read()
                if ok and _f is not None and _f.size:
                    break
                time.sleep(0.1)
            if not ok:
                cap.release()
                raise RuntimeError("Webcam is busy or blocked. Close Zoom/Teams/Camera apps, "
                                   "and allow desktop apps to use the camera in Windows settings.")
        with self.lock:
            if self.cap is not None:
                self.cap.release()
            self.cap, self.kind, self.label = cap, kind, label
            self.fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
            if self.fps > 120 or self.fps < 1:
                self.fps = 25.0
            self.width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            self.height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            self.error, self.paused = None, False

    def _loop(self):
        next_t = time.perf_counter()
        while not self._stop.is_set():
            with self.lock:
                cap, kind, fps, paused = self.cap, self.kind, self.fps, self.paused
            if cap is None or paused:
                time.sleep(0.03)
                continue
            ok, frame = cap.read()
            if not ok:
                if kind == "file":       # loop the video for the demo
                    with self.lock:
                        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                    continue
                self.error = "Camera stopped sending frames"
                time.sleep(0.2)
                continue
            # keep big videos responsive: cap the processing resolution at 1280 px wide
            if frame.shape[1] > 1280:
                s = 1280 / frame.shape[1]
                frame = cv2.resize(frame, (1280, int(frame.shape[0] * s)))
            with self.lock:
                self.frame = frame
                self.frame_id += 1
            if kind == "file":           # play files at their real speed
                next_t += 1.0 / fps
                delay = next_t - time.perf_counter()
                if delay > 0:
                    time.sleep(delay)
                else:
                    next_t = time.perf_counter()

    def latest(self):
        with self.lock:
            return (None if self.frame is None else self.frame.copy()), self.frame_id

    def seek(self, seconds: float):
        with self.lock:
            if self.cap is not None and self.kind == "file":
                self.cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, seconds) * 1000)


# =============================================================================== engine
class Engine:
    def __init__(self, source: str):
        print("Loading models ...")
        self.rec = FaceRecognizer()
        self.pipe = CrowdPipeline(self.rec, Settings(detect_every=2))
        self.det_size = config.DET_SIZE
        self.model_lock = threading.Lock()
        self.src = VideoSource()
        self.jpeg = self._placeholder("Starting ...")
        self.raw_frame = None
        self.jpeg_id = 0
        self.jobs: dict[str, dict] = {}
        self._search_rec = None
        self.enrolling = None     # {"name", "progress", "message"}
        try:
            self.src.open(source)
        except RuntimeError as e:
            print("!", e)
            vids = [p for p in sorted(config.VIDEO_DIR.glob("*")) if p.suffix.lower() in VIDEO_EXT]
            if source.isdigit() and vids:       # demo fallback: play a video instead of a black screen
                print(f"  Falling back to video: {vids[0].name}")
                self.src.open(str(vids[0]))
                self.src.error = f"Webcam unavailable, showing {vids[0].name}. {e}"
            else:
                self.jpeg = self._placeholder(str(e)[:60])
                self.src.error = str(e)
        threading.Thread(target=self._work, daemon=True).start()
        print(f"Ready. Device: {self.rec.device} | gallery: {len(self.rec.gallery)} people")

    @staticmethod
    def _placeholder(text: str) -> bytes:
        img = np.full((540, 960, 3), 24, np.uint8)
        cv2.putText(img, text, (40, 280), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (200, 200, 200), 2)
        return cv2.imencode(".jpg", img)[1].tobytes()

    def _work(self):
        last_id = -1
        while True:
            frame, fid = self.src.latest()
            if frame is None or fid == last_id:
                time.sleep(0.005)
                continue
            last_id = fid
            with self.model_lock:
                tracks = self.pipe.step(frame)
                out = self.pipe.draw(frame, tracks)
            self.raw_frame = frame
            if self.enrolling:
                e = self.enrolling
                cv2.rectangle(out, (0, 0), (out.shape[1], 46), (40, 120, 0), -1)
                cv2.putText(out, f"ENROLLING {e['name']}: {e['message']}", (14, 31),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            self.jpeg = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, 80])[1].tobytes()
            self.jpeg_id += 1

    # ---------------------------------------------------------------- enrollment
    def enroll_from_stream(self, name: str, shots: int = 8, seconds: float = 6.0):
        """Capture several frames of the largest face in view and add the person."""
        safe = "".join(c for c in name.strip() if c.isalnum() or c in "_- ").strip().replace(" ", "_")
        if not safe:
            raise ValueError("Please enter a name")
        out_dir = config.GALLERY_DIR / safe
        out_dir.mkdir(parents=True, exist_ok=True)
        start_n = len(list(out_dir.glob("*.jpg")))
        self.enrolling = {"name": safe, "progress": 0, "message": "look at the camera"}
        embs, last_id, t_end = [], -1, time.time() + seconds
        tips = ["look straight", "turn a little left", "turn a little right", "chin up slightly",
                "chin down slightly", "smile", "look straight", "almost done"]
        try:
            while len(embs) < shots and time.time() < t_end + 4:
                frame, fid = self.src.latest()
                if frame is None or fid == last_id:
                    time.sleep(0.02)
                    continue
                last_id = fid
                with self.model_lock:
                    boxes, scores, kpss = self.rec.detector.detect(frame)
                    if len(boxes) == 0:
                        self.enrolling["message"] = "no face found - step closer"
                        continue
                    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
                    i = int(areas.argmax())
                    if (boxes[i, 2] - boxes[i, 0]) < 60:
                        self.enrolling["message"] = "face too small - step closer"
                        continue
                    e = self.rec.embed_batch(frame, [kpss[i]])[0]
                if embs and max(float(e @ x) for x in embs) > 0.97:
                    continue  # nearly identical to a shot we already have
                embs.append(e)
                cv2.imwrite(str(out_dir / f"{start_n + len(embs)}.jpg"), frame)
                self.enrolling["progress"] = len(embs) / shots
                self.enrolling["message"] = f"{len(embs)}/{shots} - {tips[len(embs) % len(tips)]}"
                time.sleep(seconds / shots * 0.6)
            if len(embs) < 2:
                raise ValueError("Could not capture a clear face. Stand closer, facing the camera.")
            # consistency check: drop shots that disagree with the rest
            E = np.array(embs)
            mean = l2norm(E.mean(0))
            keep = E[(E @ mean) > 0.5] if ((E @ mean) > 0.5).sum() >= 2 else E
            with self.model_lock:
                self.rec.gallery.add(safe, keep.mean(0))
                self.rec.gallery.save()
            return {"name": safe, "shots": int(len(keep)), "gallery": self.rec.gallery.names}
        finally:
            self.enrolling = None

    def enroll_from_files(self, name: str, files: list[tuple[str, bytes]]):
        safe = "".join(c for c in name.strip() if c.isalnum() or c in "_- ").strip().replace(" ", "_")
        if not safe:
            raise ValueError("Please enter a name")
        out_dir = config.GALLERY_DIR / safe
        out_dir.mkdir(parents=True, exist_ok=True)
        embs = []
        n0 = len(list(out_dir.glob("*")))
        for k, (fname, data) in enumerate(files):
            img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
            if img is None:
                continue
            with self.model_lock:
                e = self.rec.embed_largest_face(img)
            if e is not None:
                embs.append(e)
                cv2.imwrite(str(out_dir / f"{n0 + k + 1}.jpg"), img)
        if not embs:
            raise ValueError("No face found in the uploaded photo(s)")
        with self.model_lock:
            self.rec.gallery.add(safe, np.mean(embs, 0))
            self.rec.gallery.save()
        return {"name": safe, "shots": len(embs), "gallery": self.rec.gallery.names}

    def remove_person(self, name: str, delete_photos: bool = False):
        with self.model_lock:
            self.rec.gallery.remove(name)
            self.rec.gallery.save()
        self.pipe.watchlist.discard(name)
        if delete_photos:
            d = config.GALLERY_DIR / name
            if d.is_dir():
                shutil.rmtree(d, ignore_errors=True)

    def rebuild_gallery(self):
        with self.model_lock:
            g = self.rec.enroll_folder(config.GALLERY_DIR, verbose=False)
        return g.names

    # ---------------------------------------------------------------- find person
    def start_find(self, photo: bytes, video: str, threshold: float) -> str:
        img = cv2.imdecode(np.frombuffer(photo, np.uint8), cv2.IMREAD_COLOR)
        if img is None:
            raise ValueError("Could not read the photo")
        with self.model_lock:
            q = self.rec.embed_largest_face(img)
        if q is None:
            raise ValueError("No face found in the photo")
        path = Path(video)
        if not path.is_absolute():
            path = ROOT / path
        if not path.exists():
            raise ValueError(f"Video not found: {video}")
        job_id = uuid.uuid4().hex[:8]
        job = {"id": job_id, "status": "running", "progress": 0.0, "video": path.name,
               "result": None, "error": None, "stop": False}
        self.jobs[job_id] = job

        def run():
            try:
                if self._search_rec is None:   # separate model copy: live feed keeps running
                    self._search_rec = FaceRecognizer()
                job["result"] = scan_video(
                    self._search_rec, q, str(path), threshold,
                    progress=lambda p: job.__setitem__("progress", round(p, 3)),
                    should_stop=lambda: job["stop"])
                job["status"] = "done"
                job["progress"] = 1.0
            except Exception as e:  # noqa: BLE001
                job["status"], job["error"] = "error", str(e)

        threading.Thread(target=run, daemon=True).start()
        return job_id


# =============================================================================== API
app = FastAPI(title="Crowd Face Recognition")
engine: Engine | None = None


def list_videos():
    vids = []
    for p in sorted(config.VIDEO_DIR.glob("*")):
        if p.suffix.lower() in VIDEO_EXT:
            vids.append({"name": p.name, "path": str(p.relative_to(ROOT)).replace("\\", "/"),
                         "mb": round(p.stat().st_size / 1e6, 1)})
    return vids


@app.get("/")
def index():
    return FileResponse(STATIC / "index.html")


@app.get("/stream")
def stream():
    def gen():
        last = -1
        while True:
            if engine.jpeg_id == last:
                time.sleep(0.01)
                continue
            last = engine.jpeg_id
            yield (b"--frame\r\nContent-Type: image/jpeg\r\n\r\n" + engine.jpeg + b"\r\n")
    return StreamingResponse(gen(), media_type="multipart/x-mixed-replace; boundary=frame")


@app.get("/api/state")
def state():
    s = engine.pipe.state()
    src = engine.src
    s["source"] = {"kind": src.kind, "label": src.label, "width": src.width, "height": src.height,
                   "error": src.error, "paused": src.paused}
    s["device"] = engine.rec.device
    s["settings"]["det_size"] = engine.det_size
    s["model"] = config.MODEL_PACK
    s["enrolling"] = engine.enrolling
    return s


@app.get("/api/heatmap")
def heatmap():
    return {"grid": engine.pipe.heatmap()}


@app.get("/api/videos")
def videos():
    return {"videos": list_videos()}


@app.post("/api/settings")
async def settings(body: dict):
    s = engine.pipe.s
    for k, v in body.items():
        if hasattr(s, k):
            cur = getattr(s, k)
            setattr(s, k, type(cur)(v) if not isinstance(cur, bool) else bool(v))
    if "det_size" in body:           # detector input resolution: 320 fast ... 640 accurate
        size = int(body["det_size"])
        if size in (320, 416, 480, 640, 800):
            engine.det_size = engine.rec.detector.det_size = size
    if "threshold" in body:          # new threshold: let tracks re-vote quickly
        for t in engine.pipe.tracker.tracks:
            t.votes.clear(); t.best_sim.clear(); t.n_rec = 0; t.last_rec_frame = -10**9
    return {"settings": s.__dict__}


@app.post("/api/source")
async def set_source(body: dict):
    src = str(body.get("source", "0"))
    try:
        engine.src.open(src)
    except RuntimeError as e:
        raise HTTPException(400, str(e))
    with engine.model_lock:
        engine.pipe.tracker.reset()
    return {"ok": True, "label": engine.src.label}


@app.post("/api/pause")
async def pause(body: dict):
    engine.src.paused = bool(body.get("paused", not engine.src.paused))
    return {"paused": engine.src.paused}


@app.post("/api/seek")
async def seek(body: dict):
    engine.src.seek(float(body.get("t", 0)))
    with engine.model_lock:
        engine.pipe.tracker.reset()
    return {"ok": True}


@app.post("/api/upload_video")
async def upload_video(file: UploadFile = File(...), play: bool = Form(True)):
    ext = Path(file.filename).suffix.lower()
    if ext not in VIDEO_EXT:
        raise HTTPException(400, "Unsupported video type")
    config.VIDEO_DIR.mkdir(parents=True, exist_ok=True)
    dest = config.VIDEO_DIR / Path(file.filename).name
    with open(dest, "wb") as f:
        shutil.copyfileobj(file.file, f)
    if play:
        engine.src.open(str(dest))
        with engine.model_lock:
            engine.pipe.tracker.reset()
    return {"ok": True, "path": str(dest.relative_to(ROOT)).replace("\\", "/")}


@app.post("/api/reset")
async def reset():
    with engine.model_lock:
        engine.pipe.reset_session()
    return {"ok": True}


# ----- people / enrollment
@app.get("/api/people")
def people():
    out = []
    for n in engine.rec.gallery.names:
        d = config.GALLERY_DIR / n
        photos = sorted(p for p in d.glob("*") if p.suffix.lower() in IMG_EXT) if d.is_dir() else []
        out.append({"name": n, "photos": len(photos), "watch": n in engine.pipe.watchlist})
    return {"people": out}


@app.get("/api/people/{name}/photo")
def person_photo(name: str):
    d = config.GALLERY_DIR / name
    photos = sorted(p for p in d.glob("*") if p.suffix.lower() in IMG_EXT) if d.is_dir() else []
    if not photos:
        raise HTTPException(404)
    img = cv2.imread(str(photos[0]))
    with engine.model_lock:
        boxes, _, _ = engine.rec.detector.detect(img)
    if len(boxes):
        areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        crop = engine.pipe._crop_jpeg(img, boxes[int(areas.argmax())], pad=0.3)
        if crop:
            return Response(crop, media_type="image/jpeg")
    return FileResponse(photos[0])


@app.get("/api/snapshot/{name}")
def snapshot(name: str):
    s = engine.pipe.sightings.get(name)
    if not s or not s.snapshot:
        raise HTTPException(404)
    return Response(s.snapshot, media_type="image/jpeg")


@app.post("/api/enroll/live")
def enroll_live(body: dict):
    if engine.src.kind != "webcam":
        raise HTTPException(400, "Switch the video source to the webcam first (live enrollment uses the camera).")
    if engine.enrolling:
        raise HTTPException(409, "Already enrolling someone")
    try:
        return engine.enroll_from_stream(str(body.get("name", "")), int(body.get("shots", 8)))
    except ValueError as e:
        raise HTTPException(400, str(e))


@app.post("/api/enroll/upload")
async def enroll_upload(name: str = Form(...), files: list[UploadFile] = File(...)):
    data = [(f.filename, await f.read()) for f in files]
    try:
        return engine.enroll_from_files(name, data)
    except ValueError as e:
        raise HTTPException(400, str(e))


@app.delete("/api/people/{name}")
def delete_person(name: str, photos: bool = False):
    engine.remove_person(name, photos)
    return {"gallery": engine.rec.gallery.names}


@app.post("/api/people/rebuild")
def rebuild():
    return {"gallery": engine.rebuild_gallery()}


@app.post("/api/watchlist")
async def watchlist(body: dict):
    name, on = body.get("name"), bool(body.get("on", True))
    if on:
        engine.pipe.watchlist.add(name)
        engine.pipe.last_alert.pop(name, None)   # alert immediately if already on screen
    else:
        engine.pipe.watchlist.discard(name)
    return {"watchlist": sorted(engine.pipe.watchlist)}


# ----- find this person
@app.post("/api/find")
async def find(photo: UploadFile = File(...), video: str = Form(...),
               threshold: float = Form(config.MATCH_THRESHOLD)):
    try:
        job = engine.start_find(await photo.read(), video, threshold)
    except ValueError as e:
        raise HTTPException(400, str(e))
    return {"job": job}


@app.get("/api/find/{job_id}")
def find_status(job_id: str):
    job = engine.jobs.get(job_id)
    if not job:
        raise HTTPException(404)
    return {k: v for k, v in job.items() if k != "stop"}


@app.post("/api/find/{job_id}/cancel")
def find_cancel(job_id: str):
    if job_id in engine.jobs:
        engine.jobs[job_id]["stop"] = True
    return {"ok": True}


# ----- export
@app.get("/api/export.csv")
def export_csv():
    fname = time.strftime("sightings_%Y%m%d_%H%M%S.csv")
    return Response(engine.pipe.export_csv(), media_type="text/csv",
                    headers={"Content-Disposition": f'attachment; filename="{fname}"'})


@app.get("/api/frame.jpg")
def frame_jpg():
    return Response(engine.jpeg, media_type="image/jpeg")


@app.exception_handler(Exception)
async def on_error(request, exc):
    return JSONResponse({"detail": str(exc)}, status_code=500)


def main():
    global engine
    p = argparse.ArgumentParser()
    p.add_argument("--source", default="0", help="0 = webcam, or a video path")
    p.add_argument("--port", type=int, default=8000)
    p.add_argument("--host", default="127.0.0.1", help="use 0.0.0.0 to open from a phone on the same Wi-Fi")
    p.add_argument("--model", default=None, help="buffalo_l (accurate) or buffalo_s (fast)")
    p.add_argument("--no-browser", action="store_true")
    args = p.parse_args()

    engine = Engine(args.source)
    if not args.no_browser:
        import webbrowser
        threading.Timer(1.5, lambda: webbrowser.open(f"http://localhost:{args.port}")).start()
    import uvicorn
    print(f"\n  Dashboard:  http://localhost:{args.port}\n")
    uvicorn.run(app, host=args.host, port=args.port, log_level="warning")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile app/static/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>CrowdSight · Live Face Recognition</title>
<style>
:root{
  --bg:#0b0f14; --panel:#121820; --panel2:#18202a; --line:#243040; --text:#e6edf3; --dim:#8b9bb0;
  --known:#2dd4a7; --unknown:#f87171; --watch:#fbbf24; --accent:#60a5fa; --ok:#22c55e;
  --mono: ui-monospace, "Cascadia Code", "SF Mono", Consolas, monospace;
  --sans: "Segoe UI", system-ui, -apple-system, Roboto, sans-serif;
}
*{box-sizing:border-box}
html,body{margin:0;background:var(--bg);color:var(--text);font-family:var(--sans);font-size:14px}
button,input,select{font:inherit;color:inherit}
header{display:flex;align-items:center;gap:16px;padding:10px 18px;border-bottom:1px solid var(--line);background:#0d131a;position:sticky;top:0;z-index:5}
.brand{display:flex;align-items:center;gap:10px;font-weight:700;letter-spacing:.3px;font-size:16px}
.brand .dot{width:10px;height:10px;border-radius:50%;background:var(--unknown);box-shadow:0 0 10px var(--unknown);animation:pulse 1.4s infinite}
@keyframes pulse{50%{opacity:.35}}
.brand small{color:var(--dim);font-weight:500}
.pills{display:flex;gap:8px;margin-left:auto;flex-wrap:wrap}
.pill{font-family:var(--mono);font-size:12px;padding:4px 10px;border:1px solid var(--line);border-radius:999px;color:var(--dim);background:var(--panel)}
.pill b{color:var(--text);font-weight:600}
.btn{background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:7px 12px;cursor:pointer;transition:.15s}
.btn:hover{border-color:var(--accent)}
a.btn{color:var(--text);text-decoration:none;display:inline-block}
.btn.primary{background:var(--accent);border-color:var(--accent);color:#06121f;font-weight:600}
.btn.warn{background:#3b2a07;border-color:var(--watch);color:var(--watch)}
.btn.danger{border-color:#5b2323;color:var(--unknown)}
.btn.sm{padding:4px 8px;font-size:12px}
main{display:grid;grid-template-columns:minmax(0,1fr) 380px;gap:14px;padding:14px 18px}
@media (max-width:1100px){main{grid-template-columns:1fr}}
@media (max-width:900px){header{flex-wrap:wrap;gap:8px;padding:8px 12px}.brand small{display:none}.pills{order:3;width:100%;margin-left:0}.pill{font-size:11px;padding:3px 8px;max-width:100%;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}main{padding:10px 12px}.bottom{grid-template-columns:1fr}}
.panel{background:var(--panel);border:1px solid var(--line);border-radius:12px;overflow:hidden}
.ph{display:flex;align-items:center;gap:8px;padding:10px 14px;border-bottom:1px solid var(--line);font-weight:600;font-size:13px;letter-spacing:.4px;text-transform:uppercase;color:var(--dim)}
.ph .sp{margin-left:auto;display:flex;gap:6px}
.pb{padding:12px 14px}
/* video */
.video-wrap{position:relative;background:#000;aspect-ratio:16/9;display:flex;align-items:center;justify-content:center}
.video-wrap img{width:100%;height:100%;object-fit:contain;display:block}
#heat{position:absolute;inset:0;width:100%;height:100%;pointer-events:none;opacity:0;transition:opacity .3s}
.hud{position:absolute;left:12px;top:12px;display:flex;gap:6px}
#srcErr{display:none;background:#3b2a07;border-bottom:1px solid var(--watch);color:var(--watch);padding:8px 14px;font-size:13px}
.hud span{font-family:var(--mono);font-size:12px;background:#000a;border:1px solid #fff2;padding:3px 8px;border-radius:6px}
.live-tag{color:#fff;background:#b91c1c!important;border-color:#b91c1c!important;font-weight:700}
.controls{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px;padding:12px 14px;border-top:1px solid var(--line)}
.ctl{display:flex;flex-direction:column;gap:6px}
.ctl label{font-size:12px;color:var(--dim)}
.row{display:flex;gap:6px;align-items:center;flex-wrap:wrap}
select,input[type=text],input[type=number]{background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:7px 10px;min-width:0}
input[type=range]{width:100%;accent-color:var(--accent)}
.toggle{display:flex;align-items:center;gap:8px;cursor:pointer;user-select:none;padding:4px 0}
.toggle input{display:none}
.toggle .sw{width:34px;height:19px;border-radius:999px;background:#2a3544;position:relative;transition:.15s;flex:none}
.toggle .sw:after{content:"";position:absolute;width:15px;height:15px;border-radius:50%;background:#cbd5e1;top:2px;left:2px;transition:.15s}
.toggle input:checked + .sw{background:var(--accent)}
.toggle input:checked + .sw:after{left:17px;background:#fff}
.thr-val{font-family:var(--mono);color:var(--accent);font-weight:700}
.thr-hint{font-size:11px;color:var(--dim);display:flex;justify-content:space-between}
/* stats */
.stats{display:grid;grid-template-columns:repeat(3,1fr);gap:1px;background:var(--line)}
.stat{background:var(--panel);padding:12px 10px;text-align:center}
.stat .v{font-family:var(--mono);font-size:28px;font-weight:700;line-height:1.1}
.stat .l{font-size:11px;color:var(--dim);text-transform:uppercase;letter-spacing:.5px;margin-top:2px}
.v.known{color:var(--known)} .v.unknown{color:var(--unknown)}
/* lists */
.side{display:flex;flex-direction:column;gap:14px;min-width:0}
.people{display:flex;flex-direction:column;gap:6px;max-height:300px;overflow:auto}
.person{display:flex;align-items:center;gap:10px;padding:6px;border-radius:8px;background:var(--panel2);border:1px solid transparent}
.person.here{border-color:#1f5e4d}
.person.watch{border-color:var(--watch)}
.person img{width:42px;height:42px;border-radius:8px;object-fit:cover;background:#000;flex:none}
.person .n{font-weight:600}
.person .m{font-size:12px;color:var(--dim);font-family:var(--mono)}
.person .tag{margin-left:auto;font-size:11px;padding:2px 7px;border-radius:999px;font-weight:700}
.tag.here{background:#0f3d31;color:var(--known)} .tag.gone{background:#222b36;color:var(--dim)}
.events{font-family:var(--mono);font-size:12px;max-height:190px;overflow:auto;display:flex;flex-direction:column;gap:3px}
.ev{display:flex;gap:8px;padding:3px 0;border-bottom:1px dashed #1e2833}
.ev .t{color:var(--dim)} .ev.alert{color:var(--watch)}
.empty{color:var(--dim);font-size:13px;padding:6px 0}
.bottom{display:grid;grid-template-columns:2fr 1fr;gap:14px}
@media (max-width:900px){.bottom{grid-template-columns:1fr}}
canvas.chart{width:100%;height:170px;display:block}
.legend{display:flex;gap:14px;font-size:12px;color:var(--dim)}
.legend i{display:inline-block;width:10px;height:3px;margin-right:5px;vertical-align:middle}
/* modal */
.modal{position:fixed;inset:0;background:#000b;display:none;align-items:center;justify-content:center;z-index:20;padding:16px}
.modal.open{display:flex}
.card{background:var(--panel);border:1px solid var(--line);border-radius:14px;width:min(720px,100%);max-height:90vh;overflow:auto}
.card .ph{position:sticky;top:0;background:var(--panel)}
.tabs{display:flex;gap:6px;margin-bottom:12px}
.tabs button{flex:1}
.tabs button.on{border-color:var(--accent);color:var(--accent)}
.progress{height:8px;background:var(--panel2);border-radius:999px;overflow:hidden}
.progress div{height:100%;background:var(--accent);width:0;transition:width .2s}
.gal{display:grid;grid-template-columns:repeat(auto-fill,minmax(150px,1fr));gap:10px}
.gcard{background:var(--panel2);border:1px solid var(--line);border-radius:10px;padding:8px;display:flex;flex-direction:column;gap:6px}
.gcard img{width:100%;aspect-ratio:1;object-fit:cover;border-radius:8px;background:#000}
.gcard.watch{border-color:var(--watch)}
.timeline{position:relative;height:34px;background:var(--panel2);border-radius:8px;margin:10px 0}
.timeline .seg{position:absolute;top:4px;bottom:4px;background:var(--known);border-radius:4px;min-width:4px;cursor:pointer}
.segs{display:grid;grid-template-columns:repeat(auto-fill,minmax(120px,1fr));gap:8px}
.seg-card{background:var(--panel2);border-radius:8px;padding:6px;font-family:var(--mono);font-size:12px;cursor:pointer;border:1px solid var(--line)}
.seg-card:hover{border-color:var(--known)}
.seg-card img{width:100%;border-radius:6px;display:block;margin-bottom:4px}
.msg{font-size:13px;padding:8px 10px;border-radius:8px;margin-top:8px;display:none}
.msg.ok{display:block;background:#0f3d31;color:var(--known)} .msg.err{display:block;background:#3d1414;color:var(--unknown)}
/* alert toast */
#toasts{position:fixed;right:18px;top:70px;display:flex;flex-direction:column;gap:10px;z-index:30}
.toast{display:flex;gap:12px;align-items:center;background:#2a1f05;border:2px solid var(--watch);border-radius:12px;padding:10px 14px;min-width:300px;box-shadow:0 10px 30px #000a;animation:slide .3s}
.toast img{width:56px;height:56px;border-radius:8px;object-fit:cover}
.toast b{color:var(--watch);font-size:15px}
@keyframes slide{from{transform:translateX(120%)}}
.flash{animation:flash 1s 3}
@keyframes flash{50%{box-shadow:0 0 0 4px var(--watch)}}
.kbd{font-family:var(--mono);font-size:11px;color:var(--dim)}
</style>
</head>
<body>
<header>
  <div class="brand"><span class="dot"></span>CrowdSight <small>Face Recognition in Crowded Video</small></div>
  <div class="pills">
    <span class="pill">source <b id="p-src">–</b></span>
    <span class="pill"><b id="p-fps">0</b> fps</span>
    <span class="pill">model <b id="p-model">–</b></span>
    <span class="pill">device <b id="p-dev">–</b></span>
  </div>
  <button class="btn" onclick="openModal('gallery')">👥 People</button>
  <button class="btn primary" onclick="openModal('enroll')">＋ Enroll</button>
  <button class="btn" onclick="openModal('find')">🔍 Find person</button>
</header>

<main>
  <section style="display:flex;flex-direction:column;gap:14px;min-width:0">
    <div class="panel" id="videoPanel">
      <div id="srcErr"></div>
      <div class="video-wrap">
        <img id="feed" src="/stream" alt="live feed">
        <canvas id="heat"></canvas>
        <div class="hud"><span class="live-tag" id="liveTag">● LIVE</span><span id="hudFaces">0 faces</span><span id="hudMs">–</span></div>
      </div>
      <div class="controls">
        <div class="ctl">
          <label>Video source</label>
          <div class="row">
            <select id="srcSel" style="flex:1"></select>
            <button class="btn sm" onclick="applySource()">Play</button>
            <button class="btn sm" id="pauseBtn" onclick="togglePause()">⏸</button>
          </div>
          <div class="row">
            <label class="btn sm" style="cursor:pointer">⬆ Upload video<input type="file" accept="video/*" hidden onchange="uploadVideo(this)"></label>
            <span class="kbd" id="upMsg"></span>
          </div>
        </div>
        <div class="ctl">
          <label>Match threshold <span class="thr-val" id="thrVal">0.40</span></label>
          <input type="range" id="thr" min="0.15" max="0.80" step="0.01" value="0.40">
          <div class="thr-hint"><span>◀ more matches, more mistakes</span><span>stricter ▶</span></div>
        </div>
        <div class="ctl">
          <label>View options</label>
          <label class="toggle"><input type="checkbox" id="optPrivacy"><span class="sw"></span>Privacy mode (blur unknown faces)</label>
          <label class="toggle"><input type="checkbox" id="optHeat"><span class="sw"></span>Crowd heatmap overlay</label>
          <label class="toggle"><input type="checkbox" id="optIds" checked><span class="sw"></span>Show track IDs</label>
        </div>
        <div class="ctl">
          <label>Crowd mode</label>
          <label class="toggle"><input type="checkbox" id="optTile"><span class="sw"></span>Tiny-face mode (tiling)</label>
          <div class="row"><label style="font-size:12px;color:var(--dim)">Detect every</label>
            <select id="optEvery"><option value="1">1 frame</option><option value="2">2 frames</option><option value="3">3 frames</option><option value="5">5 frames</option></select></div>
          <div class="row"><label style="font-size:12px;color:var(--dim)">Detector size</label>
            <select id="optDet"><option value="640">640 (accurate)</option><option value="480">480 (balanced)</option><option value="320">320 (fastest)</option></select></div>
          <div class="row"><button class="btn sm" onclick="resetSession()">↺ Reset session</button><a class="btn sm" href="/api/export.csv" style="text-decoration:none">⬇ Export CSV</a></div>
        </div>
      </div>
    </div>

    <div class="bottom">
      <div class="panel">
        <div class="ph">Crowd over time<span class="sp legend"><span><i style="background:var(--accent)"></i>faces</span><span><i style="background:var(--known)"></i>known</span></span></div>
        <div class="pb"><canvas class="chart" id="chart"></canvas></div>
      </div>
      <div class="panel">
        <div class="ph">Pipeline</div>
        <div class="pb" style="font-family:var(--mono);font-size:12px;line-height:1.9" id="pipe">–</div>
      </div>
    </div>
  </section>

  <aside class="side">
    <div class="panel">
      <div class="stats">
        <div class="stat"><div class="v" id="sFaces">0</div><div class="l">Faces now</div></div>
        <div class="stat"><div class="v known" id="sKnown">0</div><div class="l">Recognized</div></div>
        <div class="stat"><div class="v unknown" id="sUnknown">0</div><div class="l">Unknown</div></div>
        <div class="stat"><div class="v" id="sUnique">0</div><div class="l" title="Each recognised person once, plus unknown faces tracked for about a second or more">People seen</div></div>
        <div class="stat"><div class="v" id="sPeople">0</div><div class="l">People ID'd</div></div>
        <div class="stat"><div class="v" id="sGallery">0</div><div class="l">Enrolled</div></div>
      </div>
    </div>
    <div class="panel" id="alertPanel">
      <div class="ph">⚠ Watchlist alerts<span class="sp"><button class="btn sm" id="soundBtn" onclick="toggleSound()">🔔 on</button></span></div>
      <div class="pb" id="alerts"><div class="empty">No alerts. Mark someone as watched in 👥 People.</div></div>
    </div>
    <div class="panel">
      <div class="ph">People identified</div>
      <div class="pb"><div class="people" id="people"><div class="empty">Nobody recognized yet.</div></div></div>
    </div>
    <div class="panel">
      <div class="ph">Event log</div>
      <div class="pb"><div class="events" id="events"><div class="empty">Waiting for events…</div></div></div>
    </div>
  </aside>
</main>

<!-- Enroll -->
<div class="modal" id="m-enroll"><div class="card">
  <div class="ph">Enroll a new person<span class="sp"><button class="btn sm" onclick="closeModal()">✕</button></span></div>
  <div class="pb">
    <div class="tabs"><button class="btn on" id="tabLive" onclick="enrollTab('live')">📷 Live from camera</button><button class="btn" id="tabUp" onclick="enrollTab('up')">🖼 Upload photos</button></div>
    <div class="ctl"><label>Name</label><input type="text" id="enName" placeholder="e.g. Vivek" maxlength="40"></div>
    <div id="enLive">
      <p style="color:var(--dim);font-size:13px"><b style="color:var(--watch)">First switch Video source to Webcam.</b> The person stands alone in front of the camera and slowly turns their head while 8 photos are taken (about 6 seconds).</p>
      <div class="row"><button class="btn primary" id="enGo" onclick="enrollLive()">Start capture</button><span id="enCount" style="font-family:var(--mono);font-size:22px;color:var(--watch)"></span></div>
      <div class="progress" style="margin-top:10px"><div id="enProg"></div></div>
      <div class="kbd" id="enStatus" style="margin-top:6px"></div>
    </div>
    <div id="enUp" style="display:none">
      <p style="color:var(--dim);font-size:13px">Choose 3–10 clear photos of one person (one face per photo).</p>
      <input type="file" id="enFiles" accept="image/*" multiple>
      <div style="margin-top:10px"><button class="btn primary" onclick="enrollUpload()">Enroll from photos</button></div>
    </div>
    <div class="msg" id="enMsg"></div>
  </div>
</div></div>

<!-- Gallery -->
<div class="modal" id="m-gallery"><div class="card">
  <div class="ph">Enrolled people<span class="sp"><button class="btn sm" onclick="rebuildGallery()">↻ Rebuild from folder</button><button class="btn sm" onclick="closeModal()">✕</button></span></div>
  <div class="pb"><p class="kbd">Turn on ⚠ Watch to get an alert, sound and snapshot as soon as that person appears.</p><div class="gal" id="gal"></div></div>
</div></div>

<!-- Find -->
<div class="modal" id="m-find"><div class="card">
  <div class="ph">Find this person in a video<span class="sp"><button class="btn sm" onclick="closeModal()">✕</button></span></div>
  <div class="pb">
    <p style="color:var(--dim);font-size:13px">Upload one photo of someone. The system scans the whole video and shows every moment they appear. They don't need to be enrolled.</p>
    <div class="ctl"><label>Photo of the person</label><input type="file" id="fPhoto" accept="image/*"></div>
    <div class="ctl" style="margin-top:8px"><label>Video to search</label><select id="fVideo"></select></div>
    <div class="ctl" style="margin-top:8px"><label>Threshold <span class="thr-val" id="fThrVal">0.40</span></label><input type="range" id="fThr" min="0.2" max="0.7" step="0.01" value="0.40" oninput="fThrVal.textContent=(+this.value).toFixed(2)"></div>
    <div class="row" style="margin-top:10px"><button class="btn primary" id="fGo" onclick="startFind()">Search video</button><span class="kbd" id="fStatus"></span></div>
    <div class="progress" style="margin-top:10px"><div id="fProg"></div></div>
    <div id="fRes"></div>
    <div class="msg" id="fMsg"></div>
  </div>
</div></div>

<div id="toasts"></div>

<script>
const $ = id => document.getElementById(id);
let S = null, soundOn = true, seenAlerts = new Set(), firstState = true, settingsDirty = 0;

async function api(path, opts={}) {
  const r = await fetch(path, opts);
  let j = null; try { j = await r.json(); } catch(e) {}
  if (!r.ok) throw new Error((j && j.detail) || r.statusText);
  return j;
}
const post = (p, b) => api(p, {method:'POST', headers:{'Content-Type':'application/json'}, body:JSON.stringify(b)});
const fmtT = ts => new Date(ts*1000).toLocaleTimeString([], {hour:'2-digit',minute:'2-digit',second:'2-digit'});
const esc = s => String(s).replace(/[&<>"']/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));

// ---------------------------------------------------------------- polling
async function poll() {
  try {
    S = await api('/api/state');
    render(S);
  } catch(e) { $('p-src').textContent = 'server offline'; }
  setTimeout(poll, 400);
}
function render(s) {
  const lab = s.source.label.length > 28 ? s.source.label.slice(0, 26) + '…' : s.source.label;
  $('p-src').textContent = lab + (s.source.width ? ` ${s.source.width}×${s.source.height}` : '');
  $('p-fps').textContent = s.fps.toFixed(1);
  const err = s.source.error; $('srcErr').style.display = err ? 'block' : 'none'; if (err) $('srcErr').textContent = '⚠ ' + err;
  $('p-model').textContent = s.model; $('p-dev').textContent = s.device;
  $('sFaces').textContent = s.faces; $('sKnown').textContent = s.known; $('sUnknown').textContent = s.unknown;
  $('sUnique').textContent = s.unique_faces; $('sPeople').textContent = s.people.length; $('sGallery').textContent = s.gallery.length;
  $('hudFaces').textContent = `${s.faces} faces · ${s.known} known`;
  $('hudMs').textContent = `det ${s.ms.detect}ms · rec ${s.ms.recognize}ms`;
  $('liveTag').textContent = s.source.paused ? '❚❚ PAUSED' : (s.source.kind === 'file' ? '● VIDEO' : '● LIVE');
  $('pauseBtn').textContent = s.source.paused ? '▶' : '⏸';
  if (firstState || Date.now() - settingsDirty > 2000) syncSettings(s.settings);
  firstState = false;

  // people
  $('people').innerHTML = s.people.length ? s.people.map(p => `
    <div class="person ${p.present?'here':''} ${p.watch?'watch':''}">
      <img src="/api/snapshot/${encodeURIComponent(p.name)}?v=${Math.round(p.best_sim*1000)}" onerror="this.style.visibility='hidden'">
      <div><div class="n">${p.watch?'⚠ ':''}${esc(p.name)}</div>
      <div class="m">first ${fmtT(p.first_seen)} · sim ${p.best_sim.toFixed(2)}</div></div>
      <span class="tag ${p.present?'here':'gone'}">${p.present?'IN VIEW':p.seconds_ago+'s ago'}</span>
    </div>`).join('') : '<div class="empty">Nobody recognized yet.</div>';

  // events
  $('events').innerHTML = s.events.length ? s.events.map(e => `
    <div class="ev ${e.type==='alert'?'alert':''}"><span class="t">${fmtT(e.time)}</span>
    <span>${e.type==='alert'?'⚠ WATCHLIST: ':'✓ '}${esc(e.name)} <span class="t">#${e.track}</span></span></div>`).join('')
    : '<div class="empty">Waiting for events…</div>';

  // alerts
  $('alerts').innerHTML = s.alerts.length ? s.alerts.map(a => `
    <div class="ev alert"><span class="t">${fmtT(a.time)}</span><span>⚠ <b>${esc(a.name)}</b> spotted (track #${a.track}, sim ${a.similarity.toFixed(2)})</span></div>`).join('')
    : '<div class="empty">No alerts. Mark someone as watched in 👥 People.</div>';
  for (const a of s.alerts) {
    const key = a.time + a.name + a.track;
    if (!seenAlerts.has(key)) { seenAlerts.add(key); if (!firstAlertsLoad) fireAlert(a); }
  }
  firstAlertsLoad = false;

  // pipeline
  $('pipe').innerHTML = `
    frame → <b style="color:var(--accent)">SCRFD detect</b> ${s.ms.detect} ms<br>
    → <b style="color:var(--accent)">track</b> (IoU + motion, ByteTrack-style)<br>
    → <b style="color:var(--known)">ArcFace</b> per track ${s.ms.recognize} ms<br>
    → vote over frames → name<br>
    <span style="color:var(--dim)">threshold ${s.settings.threshold.toFixed(2)} · detect every ${s.settings.detect_every} · ${s.settings.tile?'tiling ON':'tiling off'}</span>`;
  drawChart(s.history);
}
let firstAlertsLoad = true;

function syncSettings(st) {
  $('thr').value = st.threshold; $('thrVal').textContent = st.threshold.toFixed(2);
  $('optPrivacy').checked = st.privacy; $('optTile').checked = st.tile; $('optIds').checked = st.show_ids;
  $('optEvery').value = st.detect_every;
  if (st.det_size) $('optDet').value = st.det_size;
}
async function setSetting(obj) { settingsDirty = Date.now(); try { await post('/api/settings', obj); } catch(e) { console.error(e); } }

$('thr').addEventListener('input', e => { $('thrVal').textContent = (+e.target.value).toFixed(2); settingsDirty = Date.now(); });
$('thr').addEventListener('change', e => setSetting({threshold: +e.target.value}));
$('optPrivacy').onchange = e => setSetting({privacy: e.target.checked});
$('optTile').onchange = e => setSetting({tile: e.target.checked});
$('optIds').onchange = e => setSetting({show_ids: e.target.checked});
$('optEvery').onchange = e => setSetting({detect_every: +e.target.value});
$('optDet').onchange = e => setSetting({det_size: +e.target.value});
$('optHeat').onchange = e => { $('heat').style.opacity = e.target.checked ? .75 : 0; if (e.target.checked) drawHeat(); };

// ---------------------------------------------------------------- chart
function drawChart(h) {
  const c = $('chart'), dpr = window.devicePixelRatio || 1;
  const W = c.clientWidth, H = c.clientHeight;
  c.width = W*dpr; c.height = H*dpr;
  const g = c.getContext('2d'); g.scale(dpr, dpr); g.clearRect(0,0,W,H);
  const pad = {l:30, r:8, t:8, b:18};
  const max = Math.max(4, ...h.map(x => x[1]));
  g.strokeStyle = '#243040'; g.fillStyle = '#8b9bb0'; g.font = '11px Consolas, monospace'; g.lineWidth = 1;
  for (let i=0;i<=4;i++){ const y = pad.t + (H-pad.t-pad.b)*i/4; g.beginPath(); g.moveTo(pad.l,y); g.lineTo(W-pad.r,y); g.stroke(); g.fillText(Math.round(max*(1-i/4)), 4, y+4); }
  if (h.length < 2) return;
  const t0 = h[0][0], t1 = h[h.length-1][0] || 1;
  const X = t => pad.l + (W-pad.l-pad.r) * (t - t0) / Math.max(1e-6, t1 - t0);
  const Y = v => pad.t + (H-pad.t-pad.b) * (1 - v/max);
  for (const [idx, col] of [[1,'#60a5fa'],[2,'#2dd4a7']]) {
    g.beginPath(); h.forEach((p,i) => i ? g.lineTo(X(p[0]),Y(p[idx])) : g.moveTo(X(p[0]),Y(p[idx])));
    g.strokeStyle = col; g.lineWidth = 2; g.stroke();
    if (idx===1){ g.lineTo(X(t1),Y(0)); g.lineTo(X(t0),Y(0)); g.closePath(); g.fillStyle='#60a5fa22'; g.fill(); }
  }
  g.fillStyle = '#8b9bb0'; g.fillText(`${Math.round(t1-t0)}s window`, W-90, H-4);
}

// ---------------------------------------------------------------- heatmap
async function drawHeat() {
  if (!$('optHeat').checked) return;
  try {
    const {grid} = await api('/api/heatmap');
    const c = $('heat'), img = $('feed');
    const r = img.getBoundingClientRect(), wr = c.getBoundingClientRect();
    c.width = wr.width; c.height = wr.height;
    const g = c.getContext('2d'); g.clearRect(0,0,c.width,c.height);
    // match the letterboxed image area
    const sw = S && S.source.width || 16, sh = S && S.source.height || 9;
    const scale = Math.min(c.width/sw, c.height/sh), iw = sw*scale, ih = sh*scale, ox = (c.width-iw)/2, oy = (c.height-ih)/2;
    const rows = grid.length, cols = grid[0].length, cw = iw/cols, ch = ih/rows;
    for (let y=0;y<rows;y++) for (let x=0;x<cols;x++) {
      const v = grid[y][x]; if (v < 0.03) continue;
      const hue = 240 - 240*v; g.fillStyle = `hsla(${hue},95%,55%,${0.15+0.55*v})`;
      g.fillRect(ox + x*cw, oy + y*ch, cw+1, ch+1);
    }
  } catch(e) {}
  setTimeout(drawHeat, 1000);
}

// ---------------------------------------------------------------- source
async function loadVideos() {
  const {videos} = await api('/api/videos');
  const opts = ['<option value="0">📷 Webcam (camera 0)</option>', '<option value="1">📷 Camera 1 (external)</option>']
    .concat(videos.map(v => `<option value="${esc(v.path)}">🎞 ${esc(v.name)} (${v.mb} MB)</option>`));
  $('srcSel').innerHTML = opts.join('');
  try { const st = S || await api('/api/state');
        const cur = st.source.kind === 'webcam' ? st.source.label.replace('Webcam ','') : videos.find(v => v.name === st.source.label)?.path;
        if (cur !== undefined) $('srcSel').value = cur; } catch(e) {}
  $('fVideo').innerHTML = videos.length ? videos.map(v => `<option value="${esc(v.path)}">${esc(v.name)}</option>`).join('')
    : '<option value="">No videos yet. Upload one first.</option>';
}
async function applySource() {
  try { await post('/api/source', {source: $('srcSel').value}); refreshFeed(); }
  catch(e) { alert('Could not open source: ' + e.message); }
}
async function togglePause() { await post('/api/pause', {}); }
function refreshFeed() { $('feed').src = '/stream?t=' + Date.now(); }
async function uploadVideo(inp) {
  if (!inp.files.length) return;
  const fd = new FormData(); fd.append('file', inp.files[0]); fd.append('play', 'true');
  $('upMsg').textContent = 'uploading…';
  try { const r = await api('/api/upload_video', {method:'POST', body:fd}); $('upMsg').textContent = 'playing ' + r.path.split('/').pop(); await loadVideos(); $('srcSel').value = r.path; refreshFeed(); }
  catch(e) { $('upMsg').textContent = e.message; }
  inp.value = '';
}
async function resetSession() { if (confirm('Clear the event log, sightings and charts?')) await post('/api/reset', {}); }

// ---------------------------------------------------------------- modals
function openModal(n) { $('m-'+n).classList.add('open'); if (n==='gallery') loadGallery(); if (n==='find') loadVideos(); if (n==='enroll') { $('enMsg').className='msg'; setTimeout(()=>$('enName').focus(),50);} }
function closeModal() { document.querySelectorAll('.modal').forEach(m => m.classList.remove('open')); }
document.addEventListener('keydown', e => { if (e.key === 'Escape') closeModal(); });
document.querySelectorAll('.modal').forEach(m => m.addEventListener('click', e => { if (e.target === m) closeModal(); }));
function showMsg(el, ok, text) { el.className = 'msg ' + (ok ? 'ok' : 'err'); el.textContent = text; }

// enroll
function enrollTab(t) { $('enLive').style.display = t==='live'?'':'none'; $('enUp').style.display = t==='up'?'':'none'; $('tabLive').classList.toggle('on', t==='live'); $('tabUp').classList.toggle('on', t==='up'); }
async function enrollLive() {
  const name = $('enName').value.trim(); if (!name) return showMsg($('enMsg'), false, 'Enter a name first');
  $('enGo').disabled = true; $('enMsg').className = 'msg';
  for (let i=3;i>0;i--) { $('enCount').textContent = i; await new Promise(r => setTimeout(r, 800)); }
  $('enCount').textContent = '📸';
  const timer = setInterval(() => { if (S && S.enrolling) { $('enProg').style.width = (S.enrolling.progress*100)+'%'; $('enStatus').textContent = S.enrolling.message; } }, 150);
  try {
    const r = await post('/api/enroll/live', {name, shots: 8});
    $('enProg').style.width = '100%';
    showMsg($('enMsg'), true, `✓ ${r.name} enrolled from ${r.shots} photos. Close this and watch them get recognized live!`);
    $('enName').value = '';
  } catch(e) { showMsg($('enMsg'), false, e.message); }
  clearInterval(timer); $('enGo').disabled = false; $('enCount').textContent = ''; $('enStatus').textContent = '';
  setTimeout(() => $('enProg').style.width = '0', 1500);
}
async function enrollUpload() {
  const name = $('enName').value.trim(), files = $('enFiles').files;
  if (!name || !files.length) return showMsg($('enMsg'), false, 'Enter a name and choose photos');
  const fd = new FormData(); fd.append('name', name); for (const f of files) fd.append('files', f);
  try { const r = await api('/api/enroll/upload', {method:'POST', body:fd}); showMsg($('enMsg'), true, `✓ ${r.name} enrolled from ${r.shots} photo(s).`); $('enFiles').value=''; $('enName').value=''; }
  catch(e) { showMsg($('enMsg'), false, e.message); }
}

// gallery
async function loadGallery() {
  const {people} = await api('/api/people');
  $('gal').innerHTML = people.length ? people.map(p => `
    <div class="gcard ${p.watch?'watch':''}">
      <img src="/api/people/${encodeURIComponent(p.name)}/photo" onerror="this.style.visibility='hidden'">
      <b>${esc(p.name)}</b><span class="kbd">${p.photos} photos</span>
      <div class="row">
        <button class="btn sm ${p.watch?'warn':''}" onclick="setWatch('${esc(p.name)}', ${!p.watch})">⚠ ${p.watch?'Watching':'Watch'}</button>
        <button class="btn sm danger" onclick="removePerson('${esc(p.name)}')">Remove</button>
      </div>
    </div>`).join('') : '<div class="empty">Nobody enrolled yet. Use ＋ Enroll.</div>';
}
async function setWatch(name, on) { await post('/api/watchlist', {name, on}); loadGallery(); }
async function removePerson(name) { if (!confirm(`Remove ${name} from the gallery? (photos are kept on disk)`)) return; await api('/api/people/'+encodeURIComponent(name), {method:'DELETE'}); loadGallery(); }
async function rebuildGallery() { await post('/api/people/rebuild', {}); loadGallery(); }

// find
async function startFind() {
  const photo = $('fPhoto').files[0], video = $('fVideo').value;
  if (!photo || !video) return showMsg($('fMsg'), false, 'Choose a photo and a video');
  const fd = new FormData(); fd.append('photo', photo); fd.append('video', video); fd.append('threshold', $('fThr').value);
  $('fGo').disabled = true; $('fRes').innerHTML = ''; $('fMsg').className = 'msg';
  try {
    const {job} = await api('/api/find', {method:'POST', body:fd});
    while (true) {
      await new Promise(r => setTimeout(r, 500));
      const j = await api('/api/find/' + job);
      $('fProg').style.width = (j.progress*100) + '%'; $('fStatus').textContent = `scanning ${j.video}… ${Math.round(j.progress*100)}%`;
      if (j.status === 'done') { renderFind(j.result, video); break; }
      if (j.status === 'error') throw new Error(j.error);
    }
  } catch(e) { showMsg($('fMsg'), false, e.message); }
  $('fGo').disabled = false; $('fStatus').textContent = '';
}
function renderFind(r, video) {
  if (!r.found) { $('fRes').innerHTML = `<p class="empty">Not found in this video (closest match ${r.best_similarity.toFixed(2)}). Try lowering the threshold.</p>`; return; }
  const total = r.segments.reduce((a,s) => a + (s.end - s.start), 0);
  $('fRes').innerHTML = `
    <p><b style="color:var(--known)">Found in ${r.segments.length} segment(s)</b> · on screen ≈ ${total.toFixed(1)}s of ${r.duration.toFixed(1)}s · best match ${r.best_similarity.toFixed(2)}</p>
    <div class="timeline">${r.segments.map(s => `<div class="seg" title="${s.start}s – ${s.end}s" style="left:${100*s.start/r.duration}%;width:${Math.max(.6,100*(s.end-s.start)/r.duration)}%" onclick="jumpTo('${esc(video)}',${s.start})"></div>`).join('')}</div>
    <div class="segs">${r.segments.map(s => `<div class="seg-card" onclick="jumpTo('${esc(video)}',${s.start})">${s.thumb?`<img src="data:image/jpeg;base64,${s.thumb}">`:''}▶ ${s.start.toFixed(1)}s – ${s.end.toFixed(1)}s<br>sim ${s.best_sim.toFixed(2)}</div>`).join('')}</div>
    <p class="kbd">Click a segment to play the video from that moment in the live view.</p>`;
}
async function jumpTo(video, t) {
  if (!S || S.source.label !== video.split('/').pop()) { await post('/api/source', {source: video}); }
  await post('/api/seek', {t: Math.max(0, t - 0.5)}); closeModal(); refreshFeed();
}

// alerts
let actx = null;
function beep() {
  if (!soundOn) return;
  try { actx = actx || new AudioContext(); [0, .25].forEach(d => { const o = actx.createOscillator(), g = actx.createGain(); o.type='square'; o.frequency.value = 880; g.gain.value = .08; o.connect(g); g.connect(actx.destination); o.start(actx.currentTime+d); o.stop(actx.currentTime+d+.15); }); } catch(e) {}
}
function fireAlert(a) {
  beep();
  const el = document.createElement('div'); el.className = 'toast';
  el.innerHTML = `<img src="/api/snapshot/${encodeURIComponent(a.name)}?t=${Date.now()}" onerror="this.remove()"><div><b>⚠ WATCHLIST MATCH</b><br>${esc(a.name)} detected · track #${a.track}<br><span class="kbd">${fmtT(a.time)} · similarity ${a.similarity.toFixed(2)}</span></div>`;
  $('toasts').prepend(el); setTimeout(() => el.remove(), 8000);
  $('alertPanel').classList.remove('flash'); void $('alertPanel').offsetWidth; $('alertPanel').classList.add('flash');
}
function toggleSound() { soundOn = !soundOn; $('soundBtn').textContent = soundOn ? '🔔 on' : '🔕 off'; }

loadVideos(); poll();
</script>
</body>
</html>


In [ ]:
%%writefile scripts/enroll.py
"""Day 3: build the gallery from data/gallery/<Name>/*.jpg

  python scripts/enroll.py                 # enroll everyone in data/gallery/
  python scripts/enroll.py --list          # show who is enrolled
"""
import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from src import config  # noqa: E402
from src.recognizer import Gallery, FaceRecognizer, GALLERY_FILE  # noqa: E402


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--gallery", default=str(config.GALLERY_DIR))
    p.add_argument("--list", action="store_true")
    args = p.parse_args()

    if args.list:
        g = Gallery.load()
        print(f"{len(g)} enrolled:", ", ".join(g.names) or "(none)")
        return

    rec = FaceRecognizer()
    print(f"Enrolling from {args.gallery}  (device: {rec.device})")
    g = rec.enroll_folder(Path(args.gallery))
    print(f"\nDone: {len(g)} people saved to {GALLERY_FILE}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/process_video.py
"""Days 2–3: run detection (+ recognition) on a video file or webcam.

  python scripts/process_video.py --source data/videos/crowd.mp4             # detect + recognize
  python scripts/process_video.py --source data/videos/crowd.mp4 --detect-only
  python scripts/process_video.py --source 0 --show                           # webcam, live window
  python scripts/process_video.py --source crowd.mp4 --every 3 --tile         # faster / tiny faces

Writes outputs/<name>_annotated.mp4 and prints FPS + face statistics.
Press Q in the window to stop.
"""
import argparse
import sys
import time
from collections import Counter
from pathlib import Path

import cv2

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from src import config  # noqa: E402
from src.recognizer import FaceRecognizer, draw_results  # noqa: E402


def open_source(src: str):
    cap = cv2.VideoCapture(int(src)) if src.isdigit() else cv2.VideoCapture(src)
    if not cap.isOpened():
        sys.exit(f"Could not open video source: {src}")
    return cap


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--source", required=True, help="video path, or 0 for webcam")
    p.add_argument("--detect-only", action="store_true")
    p.add_argument("--tile", action="store_true")
    p.add_argument("--every", type=int, default=config.PROCESS_EVERY, help="process every Nth frame")
    p.add_argument("--max-frames", type=int, default=0)
    p.add_argument("--threshold", type=float, default=config.MATCH_THRESHOLD)
    p.add_argument("--blur-unknown", action="store_true", help="privacy mode")
    p.add_argument("--show", action="store_true")
    p.add_argument("--no-save", action="store_true")
    args = p.parse_args()

    rec = FaceRecognizer(threshold=args.threshold)
    recognize = not args.detect_only
    if recognize and len(rec.gallery) == 0:
        print("Note: gallery is empty — run scripts/enroll.py first. Everyone will be 'Unknown'.")
    cap = open_source(args.source)
    fps_in = cap.get(cv2.CAP_PROP_FPS) or 25
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Source {W}x{H} @ {fps_in:.0f} fps | device {rec.device} | "
          f"{'detect only' if not recognize else f'{len(rec.gallery)} people enrolled'}")

    writer = None
    if not args.no_save:
        config.OUTPUT_DIR.mkdir(exist_ok=True)
        stem = "webcam" if args.source.isdigit() else Path(args.source).stem
        out_path = config.OUTPUT_DIR / f"{stem}_annotated.mp4"
        writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (W, H))

    results, n, t_model, max_faces = [], 0, 0.0, 0
    seen = Counter()
    t_start = time.perf_counter()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if n % args.every == 0:  # skipped frames reuse the last results
            t0 = time.perf_counter()
            results = rec.process(frame, tile=args.tile, recognize=recognize)
            t_model += time.perf_counter() - t0
            max_faces = max(max_faces, len(results))
            for r in results:
                if r.name != "Unknown":
                    seen[r.name] += 1
        n += 1
        out = draw_results(frame, results, blur_unknown=args.blur_unknown)
        live_fps = n / (time.perf_counter() - t_start)
        known = sum(r.name != "Unknown" for r in results)
        hud = f"Faces {len(results)} | Known {known} | {live_fps:.1f} FPS"
        cv2.rectangle(out, (0, 0), (len(hud) * 11 + 10, 30), (0, 0, 0), -1)
        cv2.putText(out, hud, (6, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        if writer:
            writer.write(out)
        if args.show:
            cv2.imshow("Crowd Face Recognition (Q to quit)", out)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
        if args.max_frames and n >= args.max_frames:
            break

    cap.release()
    if writer:
        writer.release()
    cv2.destroyAllWindows()
    total = time.perf_counter() - t_start
    calls = max(1, (n + args.every - 1) // args.every)
    print(f"\nFrames: {n} | overall {n / total:.1f} FPS | model {1000 * t_model / calls:.0f} ms/frame "
          f"| max faces in a frame: {max_faces}")
    if recognize:
        print("Recognized (frames seen):", dict(seen.most_common()) or "nobody")
    if writer:
        print(f"Saved -> {out_path}")


if __name__ == "__main__":
    main()


## 3 · Upload data
**a) Crowd video:** an `.mp4` file, saved to `data/videos/`.
**b) Gallery photos:** a `.zip` with one folder per person, for example `Vivek/1.jpg, Vivek/2.jpg, Abhishek/1.jpg …` (4–10 clear photos each, with their consent).

You can upload both in the same dialog. Or skip the gallery here and enroll people from the dashboard later.

In [ ]:
import zipfile, shutil
from pathlib import Path
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:          # running outside Colab: put files in data/ yourself
    uploaded = {}

for name in uploaded:
    p = Path(name)
    if p.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv", ".webm"}:
        shutil.move(name, f"data/videos/{p.name}")
        print("video  ->", f"data/videos/{p.name}")
    elif p.suffix.lower() == ".zip":
        with zipfile.ZipFile(name) as z:
            z.extractall("data/gallery_tmp")
        # accept both 'Person/1.jpg' and 'gallery/Person/1.jpg' layouts
        root = Path("data/gallery_tmp")
        subs = [d for d in root.iterdir() if d.is_dir()]
        if len(subs) == 1 and all(x.is_dir() for x in subs[0].iterdir()):
            root = subs[0]
        for person in root.iterdir():
            if person.is_dir():
                shutil.copytree(person, f"data/gallery/{person.name}", dirs_exist_ok=True)
                print("person ->", person.name)
        shutil.rmtree("data/gallery_tmp")
        os.remove(name)
    elif p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        shutil.move(name, f"data/images/{p.name}")
        print("image  ->", f"data/images/{p.name}")

print("\nVideos :", os.listdir("data/videos"))
print("Gallery:", [d for d in os.listdir("data/gallery") if not d.startswith('.')])

## 4 · Enroll the gallery
`buffalo_s` is the fast model and gives a smooth live view on Colab's CPU. `buffalo_l` is more accurate; use it with a GPU runtime.
The first run downloads the model (about 120–280 MB).

In [ ]:
MODEL = "buffalo_s"          # or "buffalo_l"
os.environ["FACE_MODEL"] = MODEL
!FACE_MODEL={MODEL} python scripts/enroll.py

## 5 · Start the live dashboard
This starts the FastAPI server in the background on port 8000. The source is the first uploaded video, or the video you name in `SOURCE`.

In [ ]:
import sys, threading, time, glob, importlib
sys.path.insert(0, PROJECT)
os.environ["FACE_MODEL"] = MODEL

videos = sorted(glob.glob("data/videos/*"))
SOURCE = videos[0] if videos else "0"   # Colab has no webcam, so upload a video above
print("Source:", SOURCE)

import uvicorn
import app.server as srv
importlib.reload(srv)
srv.engine = srv.Engine(SOURCE)

def _run():
    uvicorn.run(srv.app, host="0.0.0.0", port=8000, log_level="warning")

if not any(t.name == "crowdsight" for t in threading.enumerate()):
    threading.Thread(target=_run, name="crowdsight", daemon=True).start()
time.sleep(3)
print("Dashboard server running on port 8000")

## 6 · Open the dashboard
The first line opens it in a **new browser tab**, which is best for presenting. The second line embeds it below.

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_window(8000, path="/")       # opens a link in a new tab
    output.serve_kernel_port_as_iframe(8000, path="/", height=900)
except ImportError:
    print("Not in Colab: open http://localhost:8000")

## 7 · (Optional) Check the server from Python
Prints the live counts from the same API the dashboard uses.

In [ ]:
import json, urllib.request
for _ in range(3):
    s = json.load(urllib.request.urlopen("http://localhost:8000/api/state"))
    print(f"faces {s['faces']:2d} | recognised {s['known']} | people seen {s['unique_faces']} | "
          f"{s['fps']:.1f} FPS | present: {s['present']}")
    time.sleep(2)

## 8 · (Optional) Save an annotated video without the dashboard
Runs the same pipeline frame by frame and writes `outputs/<name>_annotated.mp4`, which you can download.

In [ ]:
if videos:
    !FACE_MODEL={MODEL} python scripts/process_video.py --source "{videos[0]}" --every 2 --max-frames 600
    print(os.listdir("outputs"))

---
**Credits:** InsightFace (SCRFD, ArcFace) · FastAPI · OpenCV. Test video used for academic purposes only.
**Privacy:** enroll only people who consent. Privacy mode in the dashboard blurs everyone else.